In [3]:
import os
import time
import pandas as pd
from google import genai

In [ ]:
# Configuration
API_KEY = os.getenv("GEMINI_KEY")
MODEL_NAME = "gemini-3-flash-preview"
INPUT_PATH = "data/raw/processed_original.csv"
OUTPUT_PARAPHRASE_FILE = "data/augmented/augmented_paraphrase_raw.csv"
OUTPUT_1NF_FILE = "data/augmented/gemini_paraphrase_1nf.csv"
FINAL_OUTPUT_FILE = "final_augmented_data_gemini_paraphrase.csv"
PROGRESS_FILE = "progress.txt"

BATCH_SIZE = 5
SLEEP_TIME = 1.5
MAX_RETRIES = 3
RETRY_DELAY = 5

In [ ]:
# Initialize Client
client = genai.Client(api_key=API_KEY)

In [8]:
def call_gemini(prompt_text, retries=MAX_RETRIES, delay=RETRY_DELAY):
    """
    Send prompt to Gemini model with retry mechanism.
    """
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_text,
            )
            return response.text.strip()

        except Exception as e:
            if attempt < retries - 1:
                print(f"Retrying... ({attempt + 1})")
                time.sleep(delay)
            else:
                raise e


def save_progress(index):
    """
    Save last processed index to file.
    """
    with open(PROGRESS_FILE, "w") as f:
        f.write(str(index))


def load_progress():
    """
    Load last processed index if progress file exists.
    """
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return int(f.read().strip())
    return 0


In [9]:
# Prompt Builder
def build_prompt(texts):
    """
    Construct the full prompt for batch paraphrasing.
    """
    system_prompt = "You are a data augmentation assistant."

    user_prompt = """Generate exactly 5 different paraphrases for EACH sentence below.

Rules:
- Keep the exact same meaning.
- Use Modern Standard Arabic.
- Do NOT change intent.
- Do NOT add explanations.
- Do NOT number the sentences.
- Do NOT add quotation marks.
- Do NOT add extra words.
- For each original sentence, output its 5 paraphrases in ONE line.
- Separate paraphrases using only a comma ','.
- No comma at the end.
- Separate each sentence result with a newline.

Sentences:
"""

    for i, text in enumerate(texts):
        user_prompt += f"{i + 1}. {text}\n"

    return system_prompt + "\n\n" + user_prompt

In [11]:
# Paraphrasing Pipeline
def paraphrase_dataframe(df):
    """
    Perform batch paraphrasing and save progress incrementally.
    """
    results = []
    start_index = load_progress()

    print(f"Starting from index: {start_index}")

    for i in range(start_index, len(df), BATCH_SIZE):

        batch = df["text"].iloc[i:i + BATCH_SIZE].tolist()
        prompt = build_prompt(batch)

        output = call_gemini(prompt)
        batch_results = output.split("\n")

        # Safety check to ensure model returned correct number of lines
        if len(batch_results) != len(batch):
            print(f"Warning: Mismatch in batch starting at index {i}")
            continue

        results.extend(batch_results)

        # Save intermediate results
        temp_df = df.iloc[:len(results)].copy()
        temp_df["paraphrased_text"] = results
        temp_df.to_csv(OUTPUT_PARAPHRASE_FILE, index=False, encoding="utf-8-sig")

        save_progress(i + BATCH_SIZE)

        print(f"Saved progress: {len(results)} rows")
        time.sleep(SLEEP_TIME)

    return pd.read_csv(OUTPUT_PARAPHRASE_FILE)

In [10]:
# 1NF Conversion
def convert_to_1nf(df):
    """
    Convert comma-separated paraphrases into 1NF format (one row per paraphrase).
    """
    df["paraphrased_text"] = df["paraphrased_text"].str.split(",")
    df_1nf = df.explode("paraphrased_text")
    df_1nf["paraphrased_text"] = df_1nf["paraphrased_text"].str.strip()
    df_1nf = df_1nf.reset_index(drop=True)

    df_1nf.to_csv(OUTPUT_1NF_FILE, index=False, encoding="utf-8-sig")
    return df_1nf

In [12]:
def fix_specific_rows(data, start=425, end=430):
    """
    Fix problematic rows where paraphrases were incorrectly formatted.
    """
    subset = data.iloc[start:end].copy()

    # Split using both English and Arabic commas
    subset["paraphrased_text"] = subset["paraphrased_text"].str.split("[,،]")
    subset_1nf = subset.explode("paraphrased_text")

    # Remove old rows and merge fixed ones
    data_cleaned = data.drop(data.index[start:end]).reset_index(drop=True)
    df_combined = pd.concat([subset_1nf, data_cleaned], ignore_index=True)

    df_combined.to_csv(FINAL_OUTPUT_FILE, index=False, encoding="utf-8-sig")

    return df_combined

In [ ]:
# Load dataset
df = pd.read_csv(INPUT_PATH)
df = df.iloc[100:].copy()

print(f"Total rows to process: {len(df)}")

# Step 1: Generate paraphrases
paraphrased_df = paraphrase_dataframe(df)

# Step 2: Convert to 1NF format
df_1nf = convert_to_1nf(paraphrased_df)

# Step 3: Fix specific problematic rows (optional)
final_df = fix_specific_rows(df_1nf)

print("Pipeline completed successfully.")